# 🌊 Delhi Deep Dive — Module 02: Flood Risk
## Yamuna river flood scoring for 11 Delhi districts (2010–2023)

In [ ]:
import os, time, warnings, pickle
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
warnings.filterwarnings('ignore')

OUTPUT_DIR = '../data/outputs/delhi'
os.makedirs(OUTPUT_DIR, exist_ok=True)

START, END = '2010-01-01', '2023-12-31'
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'
FLOOD_URL   = 'https://flood-api.open-meteo.com/v1/flood'
TOPO_URL    = 'https://api.opentopodata.org/v1/srtm30m'

DELHI_DISTRICTS = {
    'Central Delhi':    {'lat': 28.6508, 'lon': 77.2219},
    'East Delhi':       {'lat': 28.6271, 'lon': 77.2907},
    'New Delhi':        {'lat': 28.6139, 'lon': 77.2090},
    'North Delhi':      {'lat': 28.7041, 'lon': 77.1025},
    'North East Delhi': {'lat': 28.6921, 'lon': 77.2979},
    'North West Delhi': {'lat': 28.7219, 'lon': 77.0878},
    'Shahdara':         {'lat': 28.6735, 'lon': 77.2894},
    'South Delhi':      {'lat': 28.5244, 'lon': 77.1855},
    'South East Delhi': {'lat': 28.5677, 'lon': 77.2965},
    'South West Delhi': {'lat': 28.5672, 'lon': 77.0699},
    'West Delhi':       {'lat': 28.6541, 'lon': 77.0832},
}

YAMUNA_PROXIMITY = {
    'North East Delhi': 1.0, 'East Delhi': 0.9, 'Shahdara': 0.9,
    'North Delhi': 0.7, 'Central Delhi': 0.5, 'New Delhi': 0.3,
    'South East Delhi': 0.4, 'South Delhi': 0.2, 'West Delhi': 0.1,
    'South West Delhi': 0.1, 'North West Delhi': 0.2,
}

def score_to_category(s):
    if s <= 25: return 'LOW'
    elif s <= 50: return 'MEDIUM'
    elif s <= 75: return 'HIGH'
    return 'VERY HIGH'

def fetch_with_retry(url, params, max_retries=5):
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            wait = 5 * (2 ** attempt)
            print(f'⏳ {wait}s…', end=' ', flush=True)
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError('Max retries')

print('Flood module setup complete.')

## Fetch Rainfall + River Discharge + Elevation

In [ ]:
rain_dfs = []
for district, coords in DELHI_DISTRICTS.items():
    print(f'  {district}…', end=' ', flush=True)
    params = {'latitude': coords['lat'], 'longitude': coords['lon'],
              'start_date': START, 'end_date': END,
              'daily': 'precipitation_sum', 'timezone': 'Asia/Kolkata'}
    data = fetch_with_retry(ARCHIVE_URL, params)['daily']
    df = pd.DataFrame({'date': pd.to_datetime(data['time']),
                       'rainfall_mm': pd.to_numeric(data['precipitation_sum'], errors='coerce').fillna(0)})
    df['district'] = district
    df['lat'] = coords['lat']
    df['lon'] = coords['lon']
    rain_dfs.append(df)
    print('✓')
    time.sleep(1.5)
df_rain = pd.concat(rain_dfs, ignore_index=True)
print(f'Rainfall data: {len(df_rain):,} rows')

In [ ]:
flood_dfs = []
for district, coords in DELHI_DISTRICTS.items():
    print(f'  {district}…', end=' ', flush=True)
    try:
        params = {'latitude': coords['lat'], 'longitude': coords['lon'],
                  'start_date': START, 'end_date': END, 'daily': 'river_discharge'}
        data = fetch_with_retry(FLOOD_URL, params)['daily']
        df = pd.DataFrame({'date': pd.to_datetime(data['time']),
                           'river_discharge_m3s': pd.to_numeric(data['river_discharge'], errors='coerce').fillna(0)})
    except Exception as e:
        print(f'⚠({e})', end=' ')
        df = pd.DataFrame({'date': pd.date_range(START, END), 'river_discharge_m3s': 0.0})
    df['district'] = district
    flood_dfs.append(df)
    print('✓')
    time.sleep(1.5)
df_flood = pd.concat(flood_dfs, ignore_index=True)
print(f'River discharge data: {len(df_flood):,} rows')

In [ ]:
# Elevation
locs = '|'.join(f"{DELHI_DISTRICTS[d]['lat']},{DELHI_DISTRICTS[d]['lon']}" for d in DELHI_DISTRICTS)
elev_data = fetch_with_retry(TOPO_URL, {'locations': locs})
elevation = {d: elev_data['results'][i]['elevation'] for i, d in enumerate(DELHI_DISTRICTS)}
print('Elevations:', elevation)

# Merge + features
merged = df_rain.merge(df_flood[['district','date','river_discharge_m3s']], on=['district','date'], how='left')
merged['river_discharge_m3s'] = pd.to_numeric(merged['river_discharge_m3s'], errors='coerce').fillna(0)
merged['year'] = merged['date'].dt.year
merged = merged.sort_values(['district','date'])
merged['roll3'] = merged.groupby('district')['rainfall_mm'].transform(lambda s: s.rolling(3, min_periods=1).sum())

yearly = merged.groupby(['district','year']).agg(
    lat=('lat','first'), lon=('lon','first'),
    max_3day_rainfall_mm=('roll3','max'),
    days_above_50mm=('rainfall_mm', lambda s: (s>50).sum()),
    days_above_100mm=('rainfall_mm', lambda s: (s>100).sum()),
    peak_river_discharge_m3s=('river_discharge_m3s','max'),
).reset_index()
yearly['elevation_m'] = yearly['district'].map(elevation)
yearly['yamuna_proximity_score'] = yearly['district'].map(YAMUNA_PROXIMITY)

# Label
p80 = yearly.groupby('district')['peak_river_discharge_m3s'].quantile(0.80).reset_index()
p80.columns = ['district','p80']
yearly = yearly.merge(p80, on='district', how='left')
has_flow = yearly['peak_river_discharge_m3s'].groupby(yearly['district']).transform('max') > 0
yearly['flood_label'] = 0
yearly.loc[has_flow, 'flood_label'] = (yearly.loc[has_flow,'peak_river_discharge_m3s'] >= yearly.loc[has_flow,'p80']).astype(int)
yearly.loc[~has_flow,'flood_label'] = (yearly.loc[~has_flow,'days_above_100mm'] > 3).astype(int)
yearly = yearly.drop(columns=['p80'])

# Train
FEATURES = ['max_3day_rainfall_mm','days_above_50mm','days_above_100mm',
            'peak_river_discharge_m3s','elevation_m','yamuna_proximity_score']
train = yearly[yearly['year']<=2021].dropna(subset=FEATURES+['flood_label'])
test  = yearly[yearly['year']>=2022].dropna(subset=FEATURES+['flood_label'])

model = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42)
model.fit(train[FEATURES].astype(float), train['flood_label'].astype(int))

# Score
full = yearly.dropna(subset=FEATURES+['flood_label']).copy()
probs = model.predict_proba(full[FEATURES].astype(float))[:,1]
full['flood_prob']       = probs.round(4)
full['flood_risk_score'] = (probs * 100).round(2)
# Apply Yamuna proximity boost
prox = full['district'].map(YAMUNA_PROXIMITY).fillna(0.5)
full['flood_risk_score'] = (full['flood_risk_score'] * (0.7 + 0.3 * prox)).clip(0, 100).round(2)
full['risk_category']    = full['flood_risk_score'].apply(score_to_category)

result = full[['district','year','lat','lon','flood_label','flood_prob','flood_risk_score','risk_category']]
result.to_csv(f'{OUTPUT_DIR}/delhi_flood_scores.csv', index=False)
with open(f'{OUTPUT_DIR}/delhi_flood_model.pkl','wb') as f:
    pickle.dump(model, f)
print(f'\nSaved delhi_flood_scores.csv ({len(result)} rows)')
print(result[result['year']==2023][['district','flood_risk_score','risk_category']].to_string())